In [28]:


class Balance0:
    def __init__(self, quantity, FC):
        self.FC = FC
        self.quantity = quantity
        self.Equity = 1e6
        self.debt = 1e5 * 8.5
        self.capital = self.Equity + self.debt
        self.interest_rate = 0.15
        self.tax_rate = 0.19
        self.price = 2.5
        self.variable_unit = 2
        self.TR = self.price * self.quantity
        self.TC = self.variable_unit * self.quantity + self.FC
        self.VC = self.variable_unit * self.quantity
        self.EBIT = self.TR - self.TC
        self.I = self.interest_rate * self.debt
        self.EBT = self.EBIT - self.I
        self.Tax = self.tax_rate * self.EBT
        self.EAT = self.EBT - self.Tax
        self.ROE = self.EAT / self.Equity
    def DOL(self):
        return (self.EBIT + self.FC) / self.EBIT
    def DFL(self):
        return (self.EBIT) / (self.EBIT - self.I)
    def DTL(self):
        return (self.EBIT + self.FC) / (self.EBIT - self.I)
    def highest_risk(self):
        Q = self.quantity
        lo = 0
        hi = 1e9
        for _ in range(20000):
            mid = (lo + hi) / 2
            self.quantity = mid
            self.__init__(self.quantity, self.FC)
            f_mid = self.DTL() * self.DTL()
            if abs(f_mid) < 1e-10 or (hi - lo) < 1e-12:
                self.quantity = Q
                return mid
            if f_lo * f_mid < 0:
                hi, f_hi = mid, f_mid
            else:
                lo, f_lo = mid, f_mid
        self.quantity = Q
        return (lo + hi) / 2
    def sigma(self):
        return self.DTL() * self.DTL()


In [ ]:
class Balance:
    def __init__(self, quantity, FC):
        self.FC = FC
        self.quantity = quantity
        self.Equity = 1e6
        self.debt = 1e5 * 8.5
        self.interest_rate = 0.15
        self.tax_rate = 0.19
        self.price = 2.5
        self.variable_unit = 2

    # Using @property means these automatically update if self.quantity changes!
    @property
    def TR(self):
        return self.price * self.quantity

    @property
    def TC(self):
        return self.variable_unit * self.quantity + self.FC

    @property
    def EBIT(self):
        return self.TR - self.TC

    @property
    def I(self):
        return self.interest_rate * self.debt

    @property
    def EBT(self):
        return self.EBIT - self.I

    @property
    def Tax(self):
        return self.tax_rate * self.EBT

    @property
    def EAT(self):
        return self.EBT - self.Tax

    @property
    def ROE(self):
        return self.EAT / self.Equity

    def DOL(self):
        if self.EBIT == 0: return float('inf')
        return (self.EBIT + self.FC) / self.EBIT

    def DFL(self):
        denominator = self.EBIT - self.I
        if denominator == 0: return float('inf')
        return self.EBIT / denominator

    def DTL(self):
        denominator = self.EBIT - self.I
        if denominator == 0: return float('inf')
        return (self.EBIT + self.FC) / denominator

    def highest_risk_bisection(self, highest=True):
        """
        Finds the quantity where DTL is highest (approaches infinity).
        This happens when the denominator (EBIT - I) crosses 0.
        """
        lo = 0.0
        hi = 1e9
        
        # We define our target function: f(Q) = EBIT(Q) - I
        def f(q):
            # (Price - Variable) * Q - FC - I
            return (self.price - self.variable_unit) * q - self.FC - self.I

        f_lo = f(lo)
        
        for _ in range(100): # 100 iterations is plenty for 1e-12 precision
            mid = (lo + hi) / 2.0
            f_mid = f(mid)
            
            # If we hit 0 precisely, or our search window is tiny, we are done
            if abs(f_mid) < 1e-10 or (hi - lo) < 1e-12:
                return mid
                
            # Standard bisection check: do they have opposite signs?
            if f_lo * f_mid < 0:
                hi = mid
            else:
                lo = mid
                f_lo = f_mid # Update f_lo for the next iteration
                
        return (lo + hi) / 2.0

    def target_dtl_bisection(self, target_dtl):
        """
        Finds the quantity required to hit a specific DTL value.
        """
        lo = 0.0
        hi = 1e9
        
        def f(q):
            # DTL = (EBIT + FC) / (EBIT - I)
            # Rearranged to find root: (EBIT + FC) - target_dtl * (EBIT - I) = 0
            ebit = (self.price - self.variable_unit) * q - self.FC
            return (ebit + self.FC) - target_dtl * (ebit - self.I)

        f_lo = f(lo)
        
        for _ in range(100):
            mid = (lo + hi) / 2.0
            f_mid = f(mid)
            
            if abs(f_mid) < 1e-10 or (hi - lo) < 1e-12:
                return mid
                
            if f_lo * f_mid < 0:
                hi = mid
            else:
                lo = mid
                f_lo = f_mid
                
        return (lo + hi) / 2.0

In [31]:
q = 1e6
fc = 2.5 * 1e5
B = Balance(quantity=q, FC=fc)

In [33]:
dol = B.DOL()
dtl = B.DFL()
print(f"DOL: {dol:.2f}")
print(f"DTL: {dtl:.2f}")
dtl = B.DTL()
print(f"DTL: {dtl:.2f}")

DOL: 2.00
DTL: 2.04
DTL: 4.08


In [35]:
B.highest_risk_bisection()

755000.0

In [44]:
b9 = Balance(quantity=1.1*1e6, FC=fc)
dtl = b9.DTL()
print(f"DTL at 1,800,000 units: {dtl:.2f}")

DTL at 1,800,000 units: 3.19
